# Policy idea extraction

This Snowflake Notebook extracts individual atomic policy ideas from the AI impact survey's `GOVERNMENT_ACTION_SUGGESTION` and `ECONOMIC_IMPACT_EXPECTATION` fields.

Intermediate results are displayed in notebook cells and remain in memory. An optional write cell at the end persists the extracted ideas. The existing policy concept taxonomy is queried for reference but is not used for classification here.

Run cells from top to bottom. Set `ENABLE_OUTPUT_WRITES = True` only when you explicitly want to persist the extracted ideas.

At a high level, the workflow looks like:

1. Extract explicit policy ideas from each relevant phase 1 AI Impact Survey field ("GOVERNMENT_ACTION_SUGGESTION" and/or "ECONOMIC_IMPACT_EXPECTATION").
2. Display the extracted ideas for review.
3. Optionally persist the extracted ideas to Snowflake.


In [ ]:
# Import libraries used for hashing, JSON handling, timestamps, SQL generation, and dataframe display.
import hashlib
import json
import os
import uuid
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

# Load local environment variables for Snowflake credentials and Cortex model names.
load_dotenv()

# Set pandas display option to show full column width for better readability of long text fields.
pd.set_option('display.max_colwidth', None)

# Use the production dbt source objects currently configured for this analysis.
SOURCE_DATABASE = "TRANSFORM_ENGCA_PRD"
SOURCE_GOVOCAL_SCHEMA = "GOVOCAL"
SOURCE_AI_SCHEMA = "AI_ENGAGEMENT"
SURVEY_TABLE = "INT_GOVOCAL_AI_SURVEY"
TAXONOMY_TABLE = "STG_PHASE2_POLICY_CONCEPTS_AND_THEMES"

# Require an explicit Cortex model configuration.
CORTEX_MODEL = os.environ.get("LLM_MODEL_LOW", "")
if not CORTEX_MODEL:
    raise ValueError("Set LLM_MODEL_LOW in the local environment.")

# Record metadata for the current in-memory run.
PROMPT_VERSION = "policy-concepts-v3-set-based-sql-local"
RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

# Extract ideas from both survey fields.
EXTRACTION_FIELDS = ["government_action_suggestion", "economic_impact_expectation"]

# Set to True to write outputs.
ENABLE_OUTPUT_WRITES = False
TARGET_DATABASE = "TRANSFORM_ENGCA_DEV"
TARGET_SCHEMA = "DBT_CHOLLINGSWORTH_AI_ENGAGEMENT"
OUTPUT_TABLE = "AI_SURVEY_POLICY_IDEAS"

print(f"run_id={RUN_ID} model={CORTEX_MODEL}")

In [2]:
def query_df(sql: str, params: list | None = None) -> pd.DataFrame:
    cur.execute(sql, params or [])
    return cur.fetch_pandas_all()


def extract_structured_response(raw: str) -> str:
    payload = json.loads(raw)
    structured = payload.get("structured_output")
    if isinstance(structured, list) and structured:
        first = structured[0]
        raw_message = first.get("raw_message")
        if isinstance(raw_message, dict):
            return json.dumps(raw_message)
        if isinstance(raw_message, str):
            return raw_message
    choices = payload.get("choices")
    if isinstance(choices, list) and choices:
        first_choice = choices[0]
        messages = first_choice.get("messages")
        if isinstance(messages, str):
            return messages
        if isinstance(messages, list):
            content = "".join(msg.get("content", "") for msg in messages if isinstance(msg, dict))
            if content:
                return content
    raise ValueError("Unexpected Snowflake Cortex response shape")


def stable_id(*parts: str) -> str:
    return hashlib.md5("|".join(str(part).strip() for part in parts).encode("utf-8")).hexdigest()


def text_value(value) -> str:
    return "" if pd.isna(value) else str(value).strip()

In [ ]:
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    authenticator=os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),
    role=os.environ.get("SNOWFLAKE_ROLE", ""),
    warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE", ""),
    database=TARGET_DATABASE,
    schema=TARGET_SCHEMA,
)
cur = conn.cursor()

SOURCE_SURVEY = f'"{SOURCE_DATABASE}"."{SOURCE_GOVOCAL_SCHEMA}"."{SURVEY_TABLE}"'
SOURCE_TAXONOMY = f'"{SOURCE_DATABASE}"."{SOURCE_AI_SCHEMA}"."{TAXONOMY_TABLE}"'
TARGET_PREFIX = f'"{TARGET_DATABASE}"."{TARGET_SCHEMA}"'

print(SOURCE_SURVEY)
print(SOURCE_TAXONOMY)

In [ ]:
# Query the affinity-mapped taxonomy for reference in later steps.
taxonomy_sql = f"SELECT policy_concept_id, policy_concept, policy_concept_description, subtheme, theme FROM {SOURCE_TAXONOMY}"
taxonomy_df = query_df(taxonomy_sql).fillna("")
taxonomy_df["CONCEPT_STATUS"] = "affinity-mapped"

# Confirm that the taxonomy source contains the fields required by the pipeline.
assert {"POLICY_CONCEPT_ID", "POLICY_CONCEPT", "POLICY_CONCEPT_DESCRIPTION"}.issubset(taxonomy_df.columns)

print(f"taxonomy concepts: {len(taxonomy_df):,}")
taxonomy_df


In [ ]:
# Define the prompt for extracting explicit atomic policy recommendations.
EXTRACTION_PROMPT = """
Extract all distinct, explicit policy recommendations (ideas) from the single survey response below.
An atomic policy recommendation is a governmental or institutional action that could independently be implemented or rejected.
Split independent recommendations. Do not infer a recommendation from a condition, prediction, complaint, or consequence.
For example, saying regulation could increase employment is not a recommendation to regulate unless regulation is explicitly advocated.
Preserve the respondent's wording as much as possible and return an empty list when no explicit recommendation appears.
Return JSON only with this shape:
{"ideas": [{"idea_text": "...", "extraction_rationale": "..."}]}
""".strip()


# Provide the response schema used to request and validate structured Cortex output.
def object_schema(properties: dict, required: list[str]) -> dict:
    return {
        "type": "object",
        "properties": properties,
        "required": required,
        "additionalProperties": False,
    }


# EXTRACTION_SCHEMA = object_schema(
#     {
#         "ideas": {
#             "type": "array",
#             "items": object_schema(
#                 {
#                     "idea_text": {"type": "string"},
#                     "extraction_rationale": {"type": "string"},
#                 },
#                 ["idea_text", "extraction_rationale"],
#             ),
#         }
#     },
#     ["ideas"],
# )


# Normalize extracted ideas and check for empty ideas; malformed responses are retained for review.
def validate_extraction(payload: dict | None) -> list[dict]:
    if not isinstance(payload, dict) or not isinstance(payload.get("ideas"), list):
        raise ValueError("Extraction payload must contain an ideas list")
    ideas = []
    for item in payload["ideas"]:
        if not isinstance(item, dict) or not text_value(item.get("idea_text")):
            raise ValueError("Each extracted idea needs nonempty idea_text")
        ideas.append({"idea_text": text_value(item["idea_text"]), "extraction_rationale": text_value(item.get("extraction_rationale"))})
    return ideas


In [ ]:
# Query published survey responses that contain at least one policy-relevant answer.
survey_sql = f"""
SELECT
    survey_id,
    government_action_suggestion,
    economic_impact_expectation
FROM {SOURCE_SURVEY}
WHERE LOWER(COALESCE(publication_status, 'published')) = 'published'
  AND (
      NULLIF(TRIM(government_action_suggestion), '') IS NOT NULL
      OR NULLIF(TRIM(economic_impact_expectation), '') IS NOT NULL
  )
"""
survey_df = query_df(survey_sql)
print(f"survey responses with relevant text: {len(survey_df):,}")

# Confirm that the survey source contains the fields required by the pipeline.
assert {"SURVEY_ID", "GOVERNMENT_ACTION_SUGGESTION", "ECONOMIC_IMPACT_EXPECTATION"}.issubset(survey_df.columns)

# Build a normalized extraction input from the published survey response population.
field_sql = {
    "government_action_suggestion": "government_action_suggestion",
    "economic_impact_expectation": "economic_impact_expectation",
}
field_queries = [
    f"SELECT survey_id, '{field}' AS source_field, {field_sql[field]} AS source_text FROM source_survey WHERE NULLIF(TRIM({field_sql[field]}), '') IS NOT NULL"
    for field in EXTRACTION_FIELDS
]
survey_fields_sql = " UNION ALL ".join(field_queries)

# Run extraction for every response in one Snowflake SQL statement. JSON is validated
# in Python so one malformed model response cannot abort the whole query.
extraction_sql = f"""
WITH source_survey AS (
    {survey_sql}
),
survey_fields AS (
    {survey_fields_sql}
)
SELECT
    survey_id,
    source_field,
    source_text,
    SNOWFLAKE.CORTEX.COMPLETE(
        %s,
        ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('role', 'system', 'content', %s),
            OBJECT_CONSTRUCT('role', 'user', 'content', CONCAT('Source field: ', source_field, '\\nSurvey response:\\n', source_text))
        ),
        OBJECT_CONSTRUCT(
            'temperature', 0,
            'max_tokens', 1200
        )
    ) AS raw_cortex_response
FROM survey_fields
"""
extraction_df = query_df(
    extraction_sql,
    [CORTEX_MODEL, EXTRACTION_PROMPT],
)

# Validate and flatten each structured response while retaining malformed outputs for review.
def parse_extraction_row(row: pd.Series) -> list[dict]:
    base = {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TIMESTAMP,
        "model_name": CORTEX_MODEL,
        "prompt_version": PROMPT_VERSION,
        "survey_id": row.SURVEY_ID,
        "source_field": row.SOURCE_FIELD,
        "source_text": row.SOURCE_TEXT,
        "source_fingerprint": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, row.SOURCE_TEXT),
        "raw_cortex_response": row.RAW_CORTEX_RESPONSE,
    }
    try:
        raw = row.RAW_CORTEX_RESPONSE
        content = json.loads(extract_structured_response(raw))
        ideas = validate_extraction(content)
        if not ideas:
            return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, "NO_IDEA", row.SOURCE_TEXT), "idea_text": "", "extraction_rationale": "", "extraction_status": "no_ideas", "processing_error": None}]
        return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, idea["idea_text"]), **idea, "extraction_status": "extracted", "processing_error": None} for idea in ideas]
    except Exception as exc:
        return [{**base, "idea_id": stable_id(row.SURVEY_ID, row.SOURCE_FIELD, "INVALID", row.SOURCE_TEXT), "idea_text": "", "extraction_rationale": "", "extraction_status": "error", "processing_error": str(exc)}]


# Flatten the per-response JSON arrays into one visible dataframe of atomic ideas.
extraction_records = []
for row in extraction_df.itertuples(index=False):
    extraction_records.extend(parse_extraction_row(pd.Series(row._asdict())))
extraction_df = pd.DataFrame(extraction_records)
extraction_df.columns = [c.upper() for c in extraction_df.columns]
print(f"extraction inputs={len(survey_df):,}; extracted result rows={len(extraction_df):,}")
extraction_df

In [ ]:
# Write extracted ideas only when explicitly enabled.
if ENABLE_OUTPUT_WRITES:
    write_pandas(
        conn,
        extraction_df,
        OUTPUT_TABLE,
        database=TARGET_DATABASE,
        schema=TARGET_SCHEMA,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=True,
    )
    print(f"Wrote {len(extraction_df):,} extraction rows to {TARGET_DATABASE}.{TARGET_SCHEMA}.{OUTPUT_TABLE}.")
else:
    print("Extraction write skipped; set ENABLE_OUTPUT_WRITES = True to opt in.")


In [ ]:
# Close the local cursor and connection after all desired cells have been inspected.
cur.close()
conn.close()
print("Snowflake connection closed.")